In [ ]:
# 실행 환경·필수 패키지 확인

import sys
import subprocess
import importlib.util
from pathlib import Path

print("현재 Python:", sys.executable)

required = {
    "requests": "requests>=2.31,<3",
    "feedparser": "feedparser>=6.0.11,<7",
    "bs4": "beautifulsoup4>=4.12,<5",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("없는 패키지만 설치:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", *missing
    ])
else:
    print("필수 패키지가 이미 있습니다. 추가 설치하지 않습니다.")

root = Path(r"C:\ai_campus_main_proj\rss_news")

if not (root / "config").is_dir():
    raise FileNotFoundError(f"프로젝트 경로를 확인하세요: {root}")

print("프로젝트 경로:", root)
print("초기 확인 완료")

In [ ]:
# 공통 기능·반복 수집 시험 설정

import csv
import json
import time
import sqlite3
import hashlib
import html

from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import requests
import feedparser
from bs4 import BeautifulSoup
from IPython.display import display, HTML

TEST_DIR = root / "outputs" / "continuous_notebook_test"
TEST_DIR.mkdir(parents=True, exist_ok=True)

TEST_DB = TEST_DIR / "articles.sqlite3"

# 첫 실행은 소량으로 확인한 뒤 매체를 확대합니다.
TEST_SOURCE = "Engadget"
TEST_FEED_URL = "https://www.engadget.com/rss.xml"
BODY_LIMIT = 3

# 반복 실행 단계에서 사용할 간격입니다.
# 이 셀을 실행하는 것만으로 반복 수집되지는 않습니다.
INTERVAL_MINUTES = 10

print("시험 매체:", TEST_SOURCE)
print("실행당 본문 처리 상한:", BODY_LIMIT)
print("시험 저장소:", TEST_DB)
print("예정 확인 간격:", INTERVAL_MINUTES, "분")
print("준비 완료 — 아직 RSS·원문 요청은 하지 않았습니다.")

In [ ]:
# Engadget
# 주요 분야: 기술·전자기기·게임·디지털문화

def fetch_engadget_rss():
    response = requests.get(
        TEST_FEED_URL,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )
    print("RSS HTTP:", response.status_code)
    response.raise_for_status()

    feed = feedparser.parse(response.content)

    # HTTP 200이어도 실제 RSS인지 확인합니다.
    if not feed.version:
        raise ValueError("응답을 RSS·Atom으로 인식하지 못했습니다.")

    checked_at = datetime.now(timezone.utc).isoformat()
    unique = {}

    for entry in feed.entries:
        url = entry.get("link", "").strip()
        if not url.startswith(("https://", "http://")):
            continue

        unique.setdefault(url, {
            "source_name": TEST_SOURCE,
            "title": entry.get("title", ""),
            "url": url,
            "published": entry.get("published", ""),
            "rss_summary": entry.get("summary", ""),
            "rss_content": [
                item.get("value", "")
                for item in entry.get("content", [])
            ],
            "checked_at_utc": checked_at,
        })

    # 실행별 원본 응답을 남깁니다.
    run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    snapshot_dir = TEST_DIR / "rss_snapshots" / run_name
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    (snapshot_dir / "feed.xml").write_bytes(response.content)

    rows = list(unique.values())
    (snapshot_dir / "articles.json").write_text(
        json.dumps(rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("피드 형식:", feed.version)
    print("기사 수:", len(rows))
    print("저장 위치:", snapshot_dir)

    if feed.bozo:
        print("참고: 피드 형식 경고가 있습니다. 항목은 아래에서 확인하세요.")

    return rows


eg_articles = fetch_engadget_rss()

print("\n기사 목록 — 처음 5건")
for number, article in enumerate(eg_articles[:5], 1):
    print(f"\n{number}. {article['title']}")
    print("발행일:", article["published"])
    print("URL:", article["url"])

In [ ]:
# Engadget
# 주요 분야: 기술·전자기기·게임·디지털문화

def extract_engadget_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "extraction_method": "",
        "body_status": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    # 最初の空の領域だけを選ばず、該当する全領域を探します。
    containers = soup.select(
        "article.news-post .news-article .columns-holder"
    )

    if not containers:
        result["body_status"] = "no_body_container"
        return result

    raw_parts = []
    clean_parts = []

    for container in containers:
        # 除去前の候補を保存します。
        raw_parts.extend(
            tag.get_text(" ", strip=True)
            for tag in container.select("p, h2, h3, ul, ol, blockquote")
            if not tag.find_parent(["p", "ul", "ol", "blockquote"])
        )

        # 元の解析結果を変更しないようコピーして処理します。
        cleaned = BeautifulSoup(str(container), "html.parser")

        # 本文以外の構造が明確な要素だけ除外します。
        for tag in cleaned.select(
            "script, style, nav, aside, footer, form, button, "
            "figure, figcaption"
        ):
            tag.decompose()

        clean_parts.extend(
            tag.get_text(" ", strip=True)
            for tag in cleaned.select("p, h2, h3, ul, ol, blockquote")
            if not tag.find_parent(["p", "ul", "ol", "blockquote"])
        )

    result["body_raw"] = "\n\n".join(
        part for part in raw_parts if part
    )
    result["body_clean"] = "\n\n".join(
        part for part in clean_parts if part
    )
    result["extraction_method"] = (
        "Engadget: all columns-holder body blocks v1"
    )
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


eg_body_results = []

body_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
body_dir = TEST_DIR / "body_checks" / body_run
body_dir.mkdir(parents=True, exist_ok=True)

for number, article in enumerate(eg_articles[:BODY_LIMIT], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 70)
    print(f"[{number}] RSS 제목:", article["title"])
    print("URL:", article["url"])

    try:
        body_result = extract_engadget_body(article["url"])
    except requests.RequestException as error:
        body_result = {
            "body_status": "request_error",
            "error": str(error),
            "body_raw": "",
            "body_clean": "",
            "full_text_verified": False,
        }

    record = {**article, **body_result}
    eg_body_results.append(record)

    print("HTTP:", record.get("http_status", "응답 없음"))
    print("현재 원문 제목:", record.get("original_title", ""))
    print("상태:", record["body_status"])
    print("정제 전 글자 수:", len(record["body_raw"]))
    print("정제 후 글자 수:", len(record["body_clean"]))
    print("전문 여부: 원문 대조 필요")

    # 각 기사의 본문을 별도 파일로 저장합니다。
    (body_dir / f"{number:02d}_body.txt").write_text(
        record["body_clean"], encoding="utf-8"
    )

    display(HTML(
        '<div style="max-height:450px; overflow:auto; '
        'white-space:pre-wrap; border:1px solid #888; padding:16px;">'
        + html.escape(record["body_clean"] or "추출된 본문이 없습니다.")
        + "</div>"
    ))

(body_dir / "results.json").write_text(
    json.dumps(eg_body_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n저장 위치:", body_dir)

In [ ]:
# 기사·본문 누적 저장 및 중복 방지

def save_collection(rss_articles, body_results):
    bodies = {row["url"]: row for row in body_results}
    added = 0
    updated = 0

    with sqlite3.connect(TEST_DB) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS articles (
                url TEXT PRIMARY KEY,
                source_name TEXT,
                title TEXT,
                published TEXT,
                rss_summary TEXT,
                rss_content TEXT,
                first_seen_utc TEXT,
                last_seen_utc TEXT,
                original_title TEXT,
                body_raw TEXT,
                body_clean TEXT,
                body_status TEXT,
                extraction_method TEXT,
                body_checked_at_utc TEXT
            )
        """)

        for article in rss_articles:
            url = article["url"]
            seen_at = article["checked_at_utc"]

            exists = conn.execute(
                "SELECT 1 FROM articles WHERE url = ?", (url,)
            ).fetchone()

            conn.execute("""
                INSERT INTO articles (
                    url, source_name, title, published,
                    rss_summary, rss_content,
                    first_seen_utc, last_seen_utc, body_status
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(url) DO UPDATE SET
                    title = excluded.title,
                    published = excluded.published,
                    rss_summary = excluded.rss_summary,
                    rss_content = excluded.rss_content,
                    last_seen_utc = excluded.last_seen_utc
            """, (
                url,
                article["source_name"],
                article["title"],
                article["published"],
                article["rss_summary"],
                json.dumps(article["rss_content"], ensure_ascii=False),
                seen_at,
                seen_at,
                "pending",
            ))

            if exists:
                updated += 1
            else:
                added += 1

            body = bodies.get(url)
            if body is None:
                continue

            # 재수집 실패로 기존 본문을 지우지 않습니다.
            if body.get("body_clean"):
                conn.execute("""
                    UPDATE articles SET
                        original_title = ?,
                        body_raw = ?,
                        body_clean = ?,
                        body_status = ?,
                        extraction_method = ?,
                        body_checked_at_utc = ?
                    WHERE url = ?
                """, (
                    body.get("original_title", ""),
                    body["body_raw"],
                    body["body_clean"],
                    body["body_status"],
                    body.get("extraction_method", ""),
                    body.get("body_checked_at_utc", ""),
                    url,
                ))
            else:
                conn.execute("""
                    UPDATE articles SET
                        body_status = ?,
                        body_checked_at_utc = ?
                    WHERE url = ?
                      AND COALESCE(body_clean, '') = ''
                """, (
                    body["body_status"],
                    body.get(
                        "body_checked_at_utc",
                        datetime.now(timezone.utc).isoformat(),
                    ),
                    url,
                ))

        total = conn.execute(
            "SELECT COUNT(*) FROM articles"
        ).fetchone()[0]

        with_body = conn.execute("""
            SELECT COUNT(*) FROM articles
            WHERE LENGTH(COALESCE(body_clean, '')) > 0
        """).fetchone()[0]

        pending = conn.execute("""
            SELECT COUNT(*) FROM articles
            WHERE body_status = 'pending'
        """).fetchone()[0]

    print("이번 실행의 새 기사:", added)
    print("기존 기사 확인:", updated)
    print("전체 저장 기사:", total)
    print("본문 저장 완료:", with_body)
    print("본문 수집 대기:", pending)
    print("저장 위치:", TEST_DB)


save_collection(eg_articles, eg_body_results)

In [ ]:
# RSS 확인 → 새 기사 저장 → 대기 기사 본문 수집

def collect_once():
    print("\n수집 시작:", datetime.now().astimezone().isoformat())

    # 최신 RSS 목록을 가져와 먼저 저장합니다.
    current_articles = fetch_engadget_rss()
    save_collection(current_articles, [])

    # 이전 실행에서 남은 기사도 처리합니다.
    with sqlite3.connect(TEST_DB) as conn:
        conn.row_factory = sqlite3.Row

        pending_rows = conn.execute("""
            SELECT *
            FROM articles
            WHERE source_name = ?
              AND body_status = 'pending'
              AND LENGTH(COALESCE(body_clean, '')) = 0
            ORDER BY first_seen_utc, rowid
            LIMIT ?
        """, (TEST_SOURCE, BODY_LIMIT)).fetchall()

    results = []
    attempted_articles = []

    for number, row in enumerate(pending_rows, 1):
        if number > 1:
            time.sleep(2)

        print(f"\n본문 처리 {number}/{len(pending_rows)}:", row["title"])

        try:
            result = extract_engadget_body(row["url"])
        except requests.RequestException as error:
            result = {
                "body_status": "request_error",
                "body_raw": "",
                "body_clean": "",
                "body_checked_at_utc":
                    datetime.now(timezone.utc).isoformat(),
                "error": str(error),
            }

        results.append({"url": row["url"], **result})

        attempted_articles.append({
            "url": row["url"],
            "source_name": row["source_name"],
            "title": row["title"],
            "published": row["published"],
            "rss_summary": row["rss_summary"],
            "rss_content": json.loads(row["rss_content"] or "[]"),
            "checked_at_utc": row["last_seen_utc"],
        })

        print("상태:", result["body_status"])
        print("본문 글자 수:", len(result.get("body_clean", "")))

    if attempted_articles:
        save_collection(attempted_articles, results)
    else:
        print("\n본문 수집 대기 기사가 없습니다.")

    # 실패 이유를 포함한 본문 처리 기록도 남깁니다.
    run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    log_file = TEST_DIR / f"body_attempts_{run_name}.json"
    log_file.write_text(
        json.dumps(results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("\n이번 수집 완료")

In [ ]:
# 10분 간격으로 총 3회 수집 시험

INTERVAL_MINUTES = 10
TEST_CYCLES = 3

try:
    for cycle in range(1, TEST_CYCLES + 1):
        print(f"\n{'=' * 60}")
        print(f"수집 회차: {cycle}/{TEST_CYCLES}")

        try:
            collect_once()
        except Exception as error:
            print("이번 회차 오류:", type(error).__name__, str(error))

        if cycle < TEST_CYCLES:
            print(f"\n{INTERVAL_MINUTES}분 후 다시 확인합니다.")
            time.sleep(INTERVAL_MINUTES * 60)

    print("\n3회 수집 시험 종료")

except KeyboardInterrupt:
    print("\n사용자가 반복 수집을 중단했습니다. 저장된 결과는 유지됩니다.")

In [ ]:
# 저장된 기사 현황·목록 조회

with sqlite3.connect(TEST_DB) as conn:
    conn.row_factory = sqlite3.Row

    saved_articles = [
        dict(row)
        for row in conn.execute("""
            SELECT
                title, url, source_name, published,
                first_seen_utc, last_seen_utc,
                body_status, body_checked_at_utc,
                LENGTH(COALESCE(body_clean, '')) AS body_chars
            FROM articles
            ORDER BY first_seen_utc DESC, rowid DESC
        """).fetchall()
    ]

print("전체 저장 기사:", len(saved_articles))
print(
    "본문 저장 기사:",
    sum(row["body_chars"] > 0 for row in saved_articles)
)

print("\n상태별 기사 수")
status_counts = {}
for row in saved_articles:
    status = row["body_status"] or "상태 없음"
    status_counts[status] = status_counts.get(status, 0) + 1

for status, count in status_counts.items():
    print(f"- {status}: {count}건")

print("\n기사 목록")
for number, row in enumerate(saved_articles, 1):
    print(
        f"{number:02d}. [{row['body_status']}] "
        f"{row['title']} — 본문 {row['body_chars']}자"
    )

In [ ]:
# 기사 번호를 바꿔 저장된 본문 확인

ARTICLE_NUMBER = 1  # 위 목록에서 확인할 기사 번호

if not 1 <= ARTICLE_NUMBER <= len(saved_articles):
    print("기사 목록에 있는 번호를 입력하세요.")
else:
    selected = saved_articles[ARTICLE_NUMBER - 1]

    with sqlite3.connect(TEST_DB) as conn:
        conn.row_factory = sqlite3.Row
        article = dict(conn.execute(
            "SELECT * FROM articles WHERE url = ?",
            (selected["url"],),
        ).fetchone())

    print("매체:", article["source_name"])
    print("RSS 제목:", article["title"])
    print("원문 제목:", article["original_title"] or "미수집")
    print("URL:", article["url"])
    print("발행일:", article["published"])
    print("최초 발견:", article["first_seen_utc"])
    print("최근 RSS 확인:", article["last_seen_utc"])
    print("본문 확인:", article["body_checked_at_utc"] or "미수집")
    print("상태:", article["body_status"])
    print("추출 방식:", article["extraction_method"] or "미수집")

    text = article["body_clean"] or (
        "저장된 본문이 없습니다. 상태를 확인하세요."
    )

    display(HTML(
        '<div style="max-height:550px; overflow:auto; '
        'white-space:pre-wrap; border:1px solid #888; padding:16px;">'
        + html.escape(text)
        + "</div>"
    ))

In [ ]:
# 저장된 본문 직접 확인 — 인터넷 요청·데이터 변경 없음

import sqlite3
from pathlib import Path

check_db = Path(
    r"C:\ai_campus_main_proj\rss_news"
    r"\outputs\continuous_notebook_test\articles.sqlite3"
)

print("실행 코드: 저장 본문 직접 확인 V2")
print("확인 DB:", check_db)

if not check_db.is_file():
    print("해당 위치에 DB 파일이 없습니다.")

else:
    with sqlite3.connect(
        check_db.as_uri() + "?mode=ro", uri=True
    ) as check_conn:
        check_conn.row_factory = sqlite3.Row

        check_row = check_conn.execute("""
            SELECT title, url, body_status, body_clean
            FROM articles
            WHERE LENGTH(TRIM(COALESCE(body_clean, ''))) > 0
            ORDER BY rowid
            LIMIT 1
        """).fetchone()

    if check_row is None:
        print("이 DB에서 본문이 저장된 기사를 찾지 못했습니다.")

    else:
        print("제목:", check_row["title"])
        print("URL:", check_row["url"])
        print("상태:", check_row["body_status"])
        print("본문 글자 수:", len(check_row["body_clean"]))
        print("\n" + "=" * 60 + "\n")
        print(check_row["body_clean"])

In [ ]:
# 남은 대기 기사 본문 처리 — 한 번만 실행

previous_limit = BODY_LIMIT

try:
    BODY_LIMIT = 8
    collect_once()
finally:
    BODY_LIMIT = previous_limit

In [ ]:
# 최종 저장 상태 확인

with sqlite3.connect(TEST_DB) as conn:
    rows = conn.execute("""
        SELECT
            body_status,
            COUNT(*) AS article_count,
            SUM(
                CASE
                    WHEN LENGTH(COALESCE(body_clean, '')) > 0
                    THEN 1 ELSE 0
                END
            ) AS body_count
        FROM articles
        GROUP BY body_status
    """).fetchall()

for status, count, body_count in rows:
    print(f"{status}: {count}건 / 본문 저장 {body_count}건")

In [ ]:
# 저장된 기사 선택·본문 조회

import sqlite3
import html
from pathlib import Path
from IPython.display import display, HTML

view_db = Path(
    r"C:\ai_campus_main_proj\rss_news"
    r"\outputs\continuous_notebook_test\articles.sqlite3"
)

with sqlite3.connect(
    view_db.as_uri() + "?mode=ro", uri=True
) as view_conn:
    view_conn.row_factory = sqlite3.Row

    view_articles = [
        dict(row)
        for row in view_conn.execute("""
            SELECT
                source_name, title, original_title, url,
                published, body_status, body_clean,
                first_seen_utc, last_seen_utc,
                body_checked_at_utc
            FROM articles
            WHERE LENGTH(TRIM(COALESCE(body_clean, ''))) > 0
            ORDER BY rowid DESC
        """).fetchall()
    ]

print("본문 저장 기사:", len(view_articles), "건")

if view_articles:
    # 위젯 대신 HTML 기본 기능을 사용합니다.
    # 제목을 클릭하면 해당 기사의 저장된 본문이 펼쳐집니다.
    cards = []

    for number, article in enumerate(view_articles, 1):
        title = html.escape(article["title"] or "제목 없음")
        source = html.escape(article["source_name"] or "")
        url = html.escape(article["url"], quote=True)
        body = html.escape(article["body_clean"])

        metadata = "\n".join([
            f"매체: {article['source_name']}",
            f"RSS 제목: {article['title']}",
            f"원문 제목: {article['original_title'] or '미기록'}",
            f"발행일: {article['published']}",
            f"최초 발견: {article['first_seen_utc']}",
            f"최근 RSS 확인: {article['last_seen_utc']}",
            f"본문 확인: {article['body_checked_at_utc']}",
            f"상태: {article['body_status']}",
            f"본문 글자 수: {len(article['body_clean'])}",
        ])

        cards.append(f"""
        <details style="margin:10px 0; border:1px solid #888;
                        padding:12px;">
            <summary style="cursor:pointer; font-weight:bold;">
                {number}. [{source}] {title}
            </summary>
            <pre style="white-space:pre-wrap;">{
                html.escape(metadata)
            }</pre>
            <p><a href="{url}" target="_blank"
                  rel="noopener noreferrer">원문 열기</a></p>
            <div style="max-height:500px; overflow:auto;
                        white-space:pre-wrap; padding:12px;
                        border-top:1px solid #888;">{body}</div>
        </details>
        """)

    display(HTML("".join(cards)))

else:
    print("본문이 저장된 기사가 없습니다.")

In [ ]:
# 새 기사 유입·중복 방지 확인

def read_saved_urls():
    with sqlite3.connect(TEST_DB) as conn:
        return {
            row[0]
            for row in conn.execute(
                "SELECT url FROM articles"
            ).fetchall()
        }


# 이번 시험 시작 전 저장된 기사를 기준으로 잡습니다.
baseline_urls = read_saved_urls()

print("시험 시작 전 저장 기사:", len(baseline_urls), "건")

INTERVAL_MINUTES = 10
CHECK_CYCLES = 3

try:
    for cycle in range(1, CHECK_CYCLES + 1):
        print("\n" + "=" * 60)
        print(f"새 기사 확인: {cycle}/{CHECK_CYCLES}")

        try:
            collect_once()

            with sqlite3.connect(TEST_DB) as conn:
                conn.row_factory = sqlite3.Row
                current_rows = conn.execute("""
                    SELECT url, title, body_status, body_clean
                    FROM articles
                    ORDER BY rowid DESC
                """).fetchall()

            new_rows = [
                row for row in current_rows
                if row["url"] not in baseline_urls
            ]

            print("\n전체 저장 기사:", len(current_rows), "건")
            print("시험 시작 후 추가된 기사:", len(new_rows), "건")

            if new_rows:
                print("\n새 기사 목록")
                for row in new_rows:
                    body_chars = len(row["body_clean"] or "")
                    print(
                        f"- {row['title']}\n"
                        f"  상태: {row['body_status']} / "
                        f"본문: {body_chars}자"
                    )
            else:
                print("아직 새로 발견한 기사가 없습니다.")

        except Exception as error:
            print("이번 회차 오류:", type(error).__name__, str(error))

        if cycle < CHECK_CYCLES:
            print(f"\n{INTERVAL_MINUTES}분 후 다시 확인합니다.")
            time.sleep(INTERVAL_MINUTES * 60)

    print("\n새 기사 유입 확인 시험 종료")

except KeyboardInterrupt:
    print("\n사용자가 중단했습니다. 저장된 데이터는 유지됩니다.")

In [ ]:
# 여러 매체의 RSS 설정 통합
# 본문 추출·정제 규칙 연결은 다음 단계에서 진행합니다.

# 처음 확인한 6개 + 추가 본문 추출 사례가 있는 매체
COLLECTION_SOURCES = [
    "CNBC",
    "Fortune",
    "The Motley Fool",
    "Zacks",
    "Fox Business",
    "Business Insider",
    "BBC News",
    "The Guardian",
    "Ars Technica",
    "TechCrunch",
    "WIRED",
    "NPR",
    "ProPublica",
    "IEEE Spectrum",
    "MIT Technology Review",
    "ABC News",
    "CBS News",
    "NBC News",
    "Al Jazeera English",
    "Engadget",
]

config_files = [
    root / "config" / "feed_registry.csv",
    root / "config" / "new_feed_candidates_20261029.csv",
]

feed_lookup = {}

for config_file in config_files:
    if not config_file.is_file():
        raise FileNotFoundError(f"설정 파일이 없습니다: {config_file}")

    with config_file.open(
        encoding="utf-8-sig", newline=""
    ) as handle:
        for row in csv.DictReader(handle):
            source = row.get("source_name", "").strip()
            feed_name = row.get("feed_name", "").strip()
            feed_url = row.get("feed_url", "").strip()

            if source not in COLLECTION_SOURCES:
                continue

            # 비활성화된 설정은 제외합니다.
            if row.get("enabled", "").strip().lower() == "false":
                continue

            if not feed_url.startswith(("https://", "http://")):
                continue

            key = (source, feed_url)

            feed_lookup.setdefault(key, {
                "source_name": source,
                "feed_name": feed_name,
                "feed_url": feed_url,
            })

MULTI_FEEDS = list(feed_lookup.values())

# 설정 파일에 빠진 매체가 있는지 확인합니다.
found_sources = {feed["source_name"] for feed in MULTI_FEEDS}
missing_sources = set(COLLECTION_SOURCES) - found_sources

print("등록 매체:", len(found_sources), "개")
print("등록 피드:", len(MULTI_FEEDS), "개")

for source in COLLECTION_SOURCES:
    names = [
        feed["feed_name"]
        for feed in MULTI_FEEDS
        if feed["source_name"] == source
    ]
    print(f"- {source}: {', '.join(names) or '설정 없음'}")

if missing_sources:
    print("\n설정 확인 필요:", ", ".join(sorted(missing_sources)))

print("\n설정 읽기 완료 — 인터넷 수집은 아직 시작하지 않았습니다.")

In [ ]:
# 여러 매체 RSS 수집 → 기사 저장
# 한 피드가 실패해도 나머지 피드는 계속 처리합니다.

def collect_multi_rss():
    run_name = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )
    run_dir = TEST_DIR / "multi_rss" / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    collected = {}
    feed_results = []

    with requests.Session() as session:
        session.headers.update({
            "User-Agent": "RSSNewsCollector/1.0"
        })

        for number, feed_config in enumerate(MULTI_FEEDS, 1):
            source = feed_config["source_name"]
            feed_name = feed_config["feed_name"]
            feed_url = feed_config["feed_url"]

            print(
                f"[{number}/{len(MULTI_FEEDS)}] "
                f"{source} / {feed_name}",
                flush=True,
            )

            status = {
                **feed_config,
                "checked_at_utc":
                    datetime.now(timezone.utc).isoformat(),
                "http_status": None,
                "status": "",
                "article_count": 0,
                "error": "",
            }

            try:
                response = session.get(
                    feed_url, timeout=(8, 25)
                )
                status["http_status"] = response.status_code
                status["final_url"] = response.url

                # 원본 응답을 그대로 보존합니다.
                # 실패 응답도 진단용으로 저장합니다.
                response_file = (
                    run_dir / f"{number:02d}_response.bin"
                )
                response_file.write_bytes(response.content)

                response.raise_for_status()
                parsed = feedparser.parse(response.content)

                if not parsed.version:
                    raise ValueError(
                        "응답을 RSS·Atom으로 인식하지 못했습니다."
                    )

                status["status"] = (
                    "parsed_with_warning"
                    if parsed.bozo else "parsed"
                )
                status["parse_warning"] = (
                    str(parsed.bozo_exception)
                    if parsed.bozo else ""
                )

                checked_at = status["checked_at_utc"]
                count = 0

                for entry in parsed.entries:
                    url = entry.get("link", "").strip()

                    if not url.startswith(("https://", "http://")):
                        continue

                    count += 1

                    if url not in collected:
                        collected[url] = {
                            "source_name": source,
                            "title": entry.get("title", ""),
                            "url": url,
                            "published": entry.get(
                                "published",
                                entry.get("updated", ""),
                            ),
                            "rss_summary": entry.get("summary", ""),
                            "rss_content": [
                                item.get("value", "")
                                for item in entry.get("content", [])
                            ],
                            "checked_at_utc": checked_at,
                            "feed_refs": [],
                        }

                    # 같은 기사가 여러 피드에 실린 출처를 남깁니다.
                    collected[url]["feed_refs"].append({
                        "source_name": source,
                        "feed_name": feed_name,
                        "feed_url": feed_url,
                    })

                status["article_count"] = count
                print(
                    f"  HTTP {response.status_code} / "
                    f"기사 항목 {count}건",
                    flush=True,
                )

                if parsed.bozo:
                    print("  피드 형식 경고: 상세 기록 저장")

            except (
                requests.RequestException,
                ValueError,
            ) as error:
                status["status"] = "failed"
                status["error"] = str(error)
                print("  수집 실패:", str(error), flush=True)

            feed_results.append(status)

            if number < len(MULTI_FEEDS):
                time.sleep(2)

    articles = list(collected.values())

    (run_dir / "articles.json").write_text(
        json.dumps(articles, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (run_dir / "feed_results.json").write_text(
        json.dumps(feed_results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    # 기존 본문은 유지하고 RSS 기사 정보를 추가·갱신합니다.
    # 피드별 출처는 위의 실행별 articles.json에 보존됩니다.
    save_collection(articles, [])

    print("\n매체별 이번 수집 기사 수 — URL 기준")

    for source in COLLECTION_SOURCES:
        count = sum(
            any(
                ref["source_name"] == source
                for ref in article["feed_refs"]
            )
            for article in articles
        )
        print(f"- {source}: {count}건")

    failures = [
        row for row in feed_results
        if row["status"] == "failed"
    ]

    print("\n실패 피드:", len(failures), "개")

    for row in failures:
        print(
            f"- {row['source_name']} / {row['feed_name']}: "
            f"{row['error']}"
        )

    print("\n이번 수집의 URL 중복 제거 기사:", len(articles), "건")
    print("실행 결과 저장 위치:", run_dir)

    return articles, feed_results


multi_articles, multi_feed_results = collect_multi_rss()

In [ ]:
# 이번 RSS 수집 결과 요약
# 매체별 기사 수·발행일 범위·저장된 본문 수 확인

from email.utils import parsedate_to_datetime

def parse_feed_datetime(value):
    if not value:
        return None

    try:
        parsed = parsedate_to_datetime(value)
    except (ValueError, TypeError, OverflowError):
        try:
            parsed = datetime.fromisoformat(
                value.replace("Z", "+00:00")
            )
        except (ValueError, TypeError):
            return None

    # 시간대가 없는 값은 임의로 UTC라고 판단하지 않습니다.
    if parsed.tzinfo is None:
        return None

    return parsed.astimezone(timezone.utc)


with sqlite3.connect(TEST_DB) as conn:
    stored_bodies = {
        url: {
            "status": status,
            "body_chars": body_chars,
        }
        for url, status, body_chars in conn.execute("""
            SELECT
                url,
                body_status,
                LENGTH(COALESCE(body_clean, ''))
            FROM articles
        """).fetchall()
    }

summary_rows = []

for source in COLLECTION_SOURCES:
    source_articles = [
        article
        for article in multi_articles
        if any(
            ref["source_name"] == source
            for ref in article["feed_refs"]
        )
    ]

    dates = [
        parsed
        for article in source_articles
        if (parsed := parse_feed_datetime(
            article.get("published", "")
        )) is not None
    ]

    body_count = sum(
        stored_bodies.get(article["url"], {}).get(
            "body_chars", 0
        ) > 0
        for article in source_articles
    )

    summary_rows.append({
        "매체": source,
        "기사 수": len(source_articles),
        "본문 저장": body_count,
        "가장 오래된 발행일(UTC)":
            min(dates).strftime("%Y-%m-%d %H:%M") if dates else "미확인",
        "가장 최근 발행일(UTC)":
            max(dates).strftime("%Y-%m-%d %H:%M") if dates else "미확인",
        "날짜 해석 불가": len(source_articles) - len(dates),
    })

columns = list(summary_rows[0].keys())

header = "".join(
    f"<th>{html.escape(column)}</th>"
    for column in columns
)

table_rows = "".join(
    "<tr>"
    + "".join(
        f"<td>{html.escape(str(row[column]))}</td>"
        for column in columns
    )
    + "</tr>"
    for row in summary_rows
)

display(HTML(
    '<div style="overflow:auto;">'
    '<table border="1" cellpadding="7" '
    'style="border-collapse:collapse;">'
    f"<thead><tr>{header}</tr></thead>"
    f"<tbody>{table_rows}</tbody>"
    "</table></div>"
))

print("이번 수집의 URL 중복 제거 기사:", len(multi_articles), "건")
print("날짜 범위는 이번 RSS 항목 기준이며, 해당 기간 전체를 뜻하지 않습니다.")

In [ ]:
# 이번 Engadget RSS 중 본문이 없는 기사 확인

with sqlite3.connect(TEST_DB) as conn:
    conn.row_factory = sqlite3.Row

    missing_body = []

    for article in multi_articles:
        if not any(
            ref["source_name"] == "Engadget"
            for ref in article["feed_refs"]
        ):
            continue

        row = conn.execute("""
            SELECT
                title, url, body_status,
                first_seen_utc, last_seen_utc,
                LENGTH(COALESCE(body_clean, '')) AS body_chars
            FROM articles
            WHERE url = ?
        """, (article["url"],)).fetchone()

        if row is not None and row["body_chars"] == 0:
            missing_body.append(dict(row))

print("이번 Engadget 목록 중 본문 없는 기사:", len(missing_body), "건")

for article in missing_body:
    print("\n제목:", article["title"])
    print("URL:", article["url"])
    print("상태:", article["body_status"])
    print("최초 발견:", article["first_seen_utc"])
    print("최근 확인:", article["last_seen_utc"])

In [ ]:
collect_once()

In [ ]:
# CNBC
# 주요 분야: 증시·금융·경제·기업

def extract_cnbc_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    blocks = soup.select(".ArticleBody-articleBody")

    if not blocks:
        result["body_status"] = "no_body_container"
        return result

    raw_parts = []
    clean_parts = []

    for block in blocks:
        raw_parts.append(block.get_text("\n", strip=True))

        cleaned = BeautifulSoup(str(block), "html.parser")

        # 明確な本文外の要素だけを除去します。
        for tag in cleaned.select(
            "script, style, nav, aside, footer, form, button, "
            "figure, figcaption"
        ):
            tag.decompose()

        # 本文段落・小見出し・リストを文書順に取得します。
        for tag in cleaned.select("p, h2, h3, ul, ol, blockquote"):
            if tag.find_parent(["p", "ul", "ol", "blockquote"]):
                continue
            text = tag.get_text(" ", strip=True)
            if text:
                clean_parts.append(text)

    result["body_raw"] = "\n\n".join(raw_parts)
    result["body_clean"] = "\n\n".join(clean_parts)
    result["extraction_method"] = (
        "CNBC: ArticleBody containers v1"
    )
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


# 今回取得したRSS一覧から一般記事候補を選びます。
cnbc_candidates = [
    article
    for article in multi_articles
    if any(
        ref["source_name"] == "CNBC"
        for ref in article["feed_refs"]
    )
    and not any(
        word in article["url"].lower()
        for word in (
            "/video/",
            "live-updates",
            "/investingclub/",
            "/pro/",
        )
    )
]

cnbc_results = []

for number, article in enumerate(cnbc_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_cnbc_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "error": str(error),
        }

    cnbc_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("본문 글자 수:", len(result["body_clean"]))
    print("전문 여부: 원문 대조 필요")

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

cnbc_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
cnbc_file = TEST_DIR / f"cnbc_body_check_{cnbc_run}.json"

cnbc_file.write_text(
    json.dumps(cnbc_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n시험 결과 저장:", cnbc_file)

In [ ]:
# 확인한 매체의 본문 추출 함수를 공통 목록에 연결

BODY_EXTRACTORS = {
    "Engadget": extract_engadget_body,
    "CNBC": extract_cnbc_body,
}

# 방금 확인한 CNBC 본문 결과를 저장합니다.
save_collection(cnbc_results, cnbc_results)

print("\n본문 추출 규칙 연결 완료")
for source in BODY_EXTRACTORS:
    print("-", source)

In [ ]:
# 연결된 매체의 본문 대기열 처리
# CNBC 영상·라이브·구독 경로는 별도 검토 상태로 분리합니다.

def process_connected_bodies(limit_per_source=3):
    for source, extractor in BODY_EXTRACTORS.items():
        print("\n" + "=" * 60)
        print("본문 처리 매체:", source)

        with sqlite3.connect(TEST_DB) as conn:
            conn.row_factory = sqlite3.Row

            pending_rows = [
                dict(row)
                for row in conn.execute("""
                    SELECT *
                    FROM articles
                    WHERE source_name = ?
                      AND body_status = 'pending'
                      AND LENGTH(COALESCE(body_clean, '')) = 0
                    ORDER BY first_seen_utc, rowid
                """, (source,)).fetchall()
            ]

            eligible = []

            for row in pending_rows:
                url = row["url"].lower()
                review_status = None

                if source == "CNBC":
                    if "/video/" in url:
                        review_status = "video_review"
                    elif "live-updates" in url:
                        review_status = "live_review"
                    elif any(
                        part in url
                        for part in ("/investingclub/", "/pro/")
                    ):
                        review_status = "subscription_review"

                if review_status:
                    conn.execute("""
                        UPDATE articles
                        SET body_status = ?
                        WHERE url = ?
                          AND body_status = 'pending'
                    """, (review_status, row["url"]))
                else:
                    eligible.append(row)

        targets = eligible[:limit_per_source]
        print("이번 본문 요청:", len(targets), "건")

        for number, row in enumerate(targets, 1):
            print(f"\n[{number}/{len(targets)}] {row['title']}")

            try:
                result = extractor(row["url"])
            except requests.RequestException as error:
                result = {
                    "body_status": "request_error",
                    "body_raw": "",
                    "body_clean": "",
                    "body_checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "error": str(error),
                }

            article = {
                "source_name": row["source_name"],
                "title": row["title"],
                "url": row["url"],
                "published": row["published"],
                "rss_summary": row["rss_summary"],
                "rss_content": json.loads(
                    row["rss_content"] or "[]"
                ),
                # 本文処理ではRSSの最終確認時刻を変更しません。
                "checked_at_utc": row["last_seen_utc"],
            }

            record = {**article, **result}

            # 処理した記事ごとに保存します。
            save_collection([article], [record])

            log_name = datetime.now(timezone.utc).strftime(
                "%Y%m%dT%H%M%S_%fZ"
            )
            (TEST_DIR / f"body_attempt_{log_name}.json").write_text(
                json.dumps(record, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            print("본문 상태:", result["body_status"])
            print("본문 글자 수:", len(result.get("body_clean", "")))

            time.sleep(2)

    print("\n연결된 매체의 이번 본문 처리 완료")


process_connected_bodies(limit_per_source=3)

In [ ]:
# CNBC 본문 처리 결과 확인 — 인터넷 요청 없음

with sqlite3.connect(TEST_DB) as conn:
    conn.row_factory = sqlite3.Row

    cnbc_status_rows = conn.execute("""
        SELECT body_status, COUNT(*) AS article_count
        FROM articles
        WHERE source_name = 'CNBC'
        GROUP BY body_status
        ORDER BY body_status
    """).fetchall()

    cnbc_recent_rows = conn.execute("""
        SELECT
            title,
            url,
            body_status,
            body_checked_at_utc,
            LENGTH(COALESCE(body_clean, '')) AS body_chars,
            LENGTH(TRIM(COALESCE(body_clean, ''))) AS trimmed_chars
        FROM articles
        WHERE source_name = 'CNBC'
          AND body_checked_at_utc IS NOT NULL
        ORDER BY body_checked_at_utc DESC
        LIMIT 6
    """).fetchall()

print("CNBC 상태별 기사 수")
for row in cnbc_status_rows:
    print(f"- {row['body_status']}: {row['article_count']}건")

print("\n최근 본문 처리 기사")

for number, row in enumerate(cnbc_recent_rows, 1):
    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", row["title"])
    print("상태:", row["body_status"])
    print("본문 글자 수:", row["body_chars"])
    print("공백 제외 확인용 글자 수:", row["trimmed_chars"])
    print("본문 확인 시각:", row["body_checked_at_utc"])
    print("URL:", row["url"])

In [ ]:
# CNBC 본문 미추출 기사 진단 — DB 변경 없음

diagnostic_url = (
    "https://www.cnbc.com/2026/11/03/"
    "texas-election-could-pressure-ai-data-center-trade.html"
)

try:
    diagnostic_response = requests.get(
        diagnostic_url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    diagnostic_soup = BeautifulSoup(
        diagnostic_response.content, "html.parser"
    )

    print("HTTP:", diagnostic_response.status_code)
    print("최종 URL:", diagnostic_response.url)
    print("응답 크기:", len(diagnostic_response.content), "바이트")

    diagnostic_heading = diagnostic_soup.find("h1")
    print(
        "제목:",
        diagnostic_heading.get_text(" ", strip=True)
        if diagnostic_heading else "찾지 못함",
    )

    print("\n본문 영역 확인")
    for selector in (
        ".ArticleBody-articleBody",
        ".group",
        '[itemprop="articleBody"]',
        "article",
    ):
        blocks = diagnostic_soup.select(selector)
        paragraph_count = sum(
            len(block.select("p")) for block in blocks
        )
        print(
            f"- {selector}: 영역 {len(blocks)}개 / "
            f"내부 문단 {paragraph_count}개"
        )

    print("\n구조화 데이터 본문 확인")
    structured_bodies = []

    def find_article_bodies(value):
        if isinstance(value, dict):
            body = value.get("articleBody")
            if isinstance(body, str) and body.strip():
                structured_bodies.append(body)

            for nested in value.values():
                find_article_bodies(nested)

        elif isinstance(value, list):
            for nested in value:
                find_article_bodies(nested)

    for script in diagnostic_soup.select(
        'script[type="application/ld+json"]'
    ):
        try:
            find_article_bodies(
                json.loads(script.string or script.get_text())
            )
        except (ValueError, TypeError):
            continue

    print("articleBody 필드:", len(structured_bodies), "개")
    for number, body in enumerate(structured_bodies, 1):
        print(f"- 후보 {number}: {len(body)}자")

    # 스크립트를 제외한 화면 텍스트에서 안내 문구를 확인합니다.
    diagnostic_visible = BeautifulSoup(
        diagnostic_response.content, "html.parser"
    )
    for tag in diagnostic_visible.select("script, style"):
        tag.decompose()

    visible_text = diagnostic_visible.get_text(" ", strip=True)

    markers = (
        "subscribe to continue",
        "sign in to continue",
        "already a subscriber",
        "access denied",
        "verify you are human",
    )

    detected = [
        marker for marker in markers
        if marker in visible_text.lower()
    ]

    print("\n안내 문구:", detected or "지정한 문구는 발견되지 않음")
    print("※ 문구 발견만으로 구독·차단 여부를 확정하지 않습니다.")

except requests.RequestException as error:
    print("원문 요청 오류:", str(error))

In [ ]:
# 확인된 CNBC Pro 기사를 구독 검토 상태로 기록

subscription_url = (
    "https://www.cnbc.com/2026/11/03/"
    "texas-election-could-pressure-ai-data-center-trade.html"
)

with sqlite3.connect(TEST_DB) as conn:
    cursor = conn.execute("""
        UPDATE articles
        SET body_status = 'subscription_review'
        WHERE url = ?
          AND LENGTH(COALESCE(body_clean, '')) = 0
    """, (subscription_url,))

print("구독 검토 상태로 변경:", cursor.rowcount, "건")

In [ ]:
# CNBC
# 주요 분야: 증시·금융·경제·기업

def extract_cnbc_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    blocks = soup.select(".ArticleBody-articleBody")

    raw_parts = []
    clean_parts = []

    for block in blocks:
        raw_parts.append(block.get_text("\n", strip=True))

        cleaned = BeautifulSoup(str(block), "html.parser")

        # 본문 밖의 구조가 명확한 요소만 제거합니다.
        for tag in cleaned.select(
            "script, style, nav, aside, footer, form, button, "
            "figure, figcaption"
        ):
            tag.decompose()

        # 본문 요소를 순서대로 가져오고 중첩 중복을 피합니다.
        for tag in cleaned.select("p, h2, h3, ul, ol, blockquote"):
            if tag.find_parent(["p", "ul", "ol", "blockquote"]):
                continue

            text = tag.get_text(" ", strip=True)
            if text:
                clean_parts.append(text)

    result["body_raw"] = "\n\n".join(raw_parts)
    result["body_clean"] = "\n\n".join(clean_parts)
    result["extraction_method"] = (
        "CNBC: ArticleBody containers v2"
    )

    # 메뉴의 CNBC Pro 표기와 구독 장벽을 구분합니다.
    # 이번에 확인한 구독 안내 문구 조합을 탐지합니다.
    visible = BeautifulSoup(response.content, "html.parser")

    for tag in visible.select("script, style, nav, header, footer"):
        tag.decompose()

    visible_text = " ".join(
        visible.get_text(" ", strip=True).lower().split()
    )

    confirmed_pro_notice = (
        "read this article and get essential investing tools"
        in visible_text
        and "when you become a pro member you unlock"
        in visible_text
    )

    if confirmed_pro_notice:
        result["body_status"] = "subscription_review"
        result["review_reason"] = "CNBC Pro 구독 장벽 문구 확인"
    elif result["body_clean"]:
        result["body_status"] = "candidate_review"
    elif blocks:
        result["body_status"] = "no_body"
    else:
        result["body_status"] = "no_body_container"

    return result


# 공통 목록도 새 함수로 갱신해야 실제 처리에 반영됩니다.
BODY_EXTRACTORS["CNBC"] = extract_cnbc_body

print("CNBC 본문 함수와 공통 연결 목록 갱신 완료")
print("아직 인터넷 요청은 하지 않았습니다.")

In [ ]:
# BBC News
# 주요 분야: 일반뉴스·국제·경제·기술

def extract_bbc_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    article = soup.find("article")
    if article is None:
        result["body_status"] = "no_body_container"
        return result

    result["body_raw"] = article.get_text("\n", strip=True)

    cleaned = BeautifulSoup(str(article), "html.parser")

    # 明確な本文外の要素を除去します。
    for tag in cleaned.select(
        "script, style, nav, aside, footer, form, button, "
        "figure, figcaption"
    ):
        tag.decompose()

    # 本文専用ブロックがあれば優先します。
    text_blocks = cleaned.select('[data-component="text-block"]')

    if text_blocks:
        parts = [
            block.get_text(" ", strip=True)
            for block in text_blocks
        ]
        method = "BBC: text-block v1"
    else:
        # 以前に本文表示が成功した方式へ切り替えます。
        parts = [
            tag.get_text(" ", strip=True)
            for tag in cleaned.select("p, h2, h3")
        ]
        method = "BBC: article paragraphs fallback v1"

    result["body_clean"] = "\n\n".join(
        part for part in parts if part
    )
    result["extraction_method"] = method
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


bbc_candidates = [
    article
    for article in multi_articles
    if any(
        ref["source_name"] == "BBC News"
        for ref in article["feed_refs"]
    )
    and "/news/articles/" in article["url"]
]

bbc_results = []

for number, article in enumerate(bbc_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_bbc_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "body_checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "error": str(error),
        }

    bbc_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("추출 방식:", result.get("extraction_method", ""))
    print("본문 글자 수:", len(result["body_clean"]))

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

bbc_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
bbc_file = TEST_DIR / f"bbc_body_check_{bbc_run}.json"

bbc_file.write_text(
    json.dumps(bbc_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n확인한 기사:", len(bbc_results), "건")
print("시험 결과 저장:", bbc_file)

In [ ]:
# BBC News
# 주요 분야: 일반뉴스·국제·경제·기술

import re

def clean_bbc_text(text):
    # 이번에 확인한 안내 문단만 제거합니다.
    exact_notices = {
        "live: follow latest updates",
        "results: the latest from around the country",
        "reaction: ecstatic scenes and historic moments",
        "guide: what are the us midterms?",
        "newsletter: sign up for anthony zurcher's analysis in your inbox",
        "related topics",
    }

    def normalize(value):
        value = value.replace("’", "'").replace("\u00a0", " ")
        return " ".join(value.split()).strip().casefold()

    kept = []
    removed = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        normalized = normalize(paragraph)

        is_exact_notice = normalized in exact_notices

        # 해당 기자의 뉴스레터 안내 문단만 식별합니다.
        is_newsletter_promotion = (
            normalized.startswith(
                "follow the twists and turns of trump's second term"
            )
            and "us politics unspun newsletter" in normalized
        )

        # 가입 안내가 별도 문단으로 추출되는 경우도 처리합니다.
        is_signup_notice = normalized in {
            "readers in the uk can sign up here.",
            "those outside the uk can sign up here.",
            (
                "readers in the uk can sign up here. "
                "those outside the uk can sign up here."
            ),
        }

        if (
            is_exact_notice
            or is_newsletter_promotion
            or is_signup_notice
        ):
            removed.append(paragraph)
        else:
            kept.append(paragraph)

    return "\n\n".join(kept), removed


# 기존 추출 함수를 한 번만 보관합니다.
# 이 셀을 다시 실행해도 함수가 중첩되지 않습니다.
if "_bbc_extractor_before_cleanup" not in globals():
    _bbc_extractor_before_cleanup = extract_bbc_body


def extract_bbc_body(url):
    result = _bbc_extractor_before_cleanup(url)

    before = result.get("body_clean", "")
    after, removed = clean_bbc_text(before)

    result["body_before_cleanup"] = before
    result["body_clean"] = after
    result["removed_paragraphs"] = removed
    result["cleanup_rule_version"] = "bbc_notices_v1"

    if before:
        result["body_status"] = (
            "candidate_review" if after else "no_body"
        )

    return result


# 이미 받은 표본을 다시 요청하지 않고 정제합니다.
for number, record in enumerate(bbc_results, 1):
    before = record.get(
        "body_before_cleanup",
        record.get("body_clean", ""),
    )
    after, removed = clean_bbc_text(before)

    record["body_before_cleanup"] = before
    record["body_clean"] = after
    record["removed_paragraphs"] = removed
    record["cleanup_rule_version"] = "bbc_notices_v1"

    if before:
        record["body_status"] = (
            "candidate_review" if after else "no_body"
        )

    print(f"\n[{number}] {record['title']}")
    print("정제 전:", len(before), "자")
    print("정제 후:", len(after), "자")
    print("제거한 문단:", len(removed), "개")

    for paragraph in removed:
        print("-", paragraph)

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">정제 본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(after or "저장할 본문이 없습니다.")
        + "</div></details>"
    ))

cleanup_run = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)
cleanup_file = TEST_DIR / f"bbc_cleanup_{cleanup_run}.json"

cleanup_file.write_text(
    json.dumps(bbc_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n정제 결과 저장:", cleanup_file)
print("BBC 공통 수집 연결·DB 저장은 아직 하지 않았습니다.")

In [ ]:
# BBC 추출·정제 함수를 공통 목록에 연결하고 표본 저장

BODY_EXTRACTORS["BBC News"] = extract_bbc_body

save_collection(bbc_results, bbc_results)

print("\n연결된 본문 추출 매체")
for source in BODY_EXTRACTORS:
    print("-", source)

print("\nBBC 저장 상태")

with sqlite3.connect(TEST_DB) as conn:
    rows = conn.execute("""
        SELECT body_status, COUNT(*)
        FROM articles
        WHERE source_name = 'BBC News'
        GROUP BY body_status
        ORDER BY body_status
    """).fetchall()

for status, count in rows:
    print(f"- {status}: {count}건")

In [ ]:
# 연결된 매체의 본문 대기열 처리
# 영상·라이브·구독 경로는 별도 검토 상태로 분리합니다.

from urllib.parse import urlsplit

def classify_body_request(source, url):
    path = urlsplit(url).path.lower()

    if source == "CNBC":
        if "/video/" in path:
            return "video_review"
        if "live-updates" in path:
            return "live_review"
        if any(
            part in path
            for part in ("/investingclub/", "/pro/")
        ):
            return "subscription_review"

    if source == "BBC News":
        if "/videos/" in path or "/av/" in path:
            return "video_review"
        if "/live/" in path:
            return "live_review"

        # 현재 검증한 일반 기사 경로만 자동 처리합니다.
        if "/news/articles/" not in path:
            return "type_review"

    return None


def process_connected_bodies(limit_per_source=3):
    for source, extractor in BODY_EXTRACTORS.items():
        print("\n" + "=" * 60)
        print("본문 처리 매체:", source)

        with sqlite3.connect(TEST_DB) as conn:
            conn.row_factory = sqlite3.Row

            pending_rows = [
                dict(row)
                for row in conn.execute("""
                    SELECT *
                    FROM articles
                    WHERE source_name = ?
                      AND body_status = 'pending'
                      AND LENGTH(COALESCE(body_clean, '')) = 0
                    ORDER BY first_seen_utc, rowid
                """, (source,)).fetchall()
            ]

            eligible = []
            classified_counts = {}

            for row in pending_rows:
                review_status = classify_body_request(
                    source, row["url"]
                )

                if review_status:
                    conn.execute("""
                        UPDATE articles
                        SET body_status = ?
                        WHERE url = ?
                          AND body_status = 'pending'
                    """, (review_status, row["url"]))

                    classified_counts[review_status] = (
                        classified_counts.get(review_status, 0) + 1
                    )
                else:
                    eligible.append(row)

        for status, count in classified_counts.items():
            print(f"별도 검토 분류: {status} / {count}건")

        targets = eligible[:limit_per_source]
        print("이번 본문 요청:", len(targets), "건")

        for number, row in enumerate(targets, 1):
            print(f"\n[{number}/{len(targets)}] {row['title']}")

            try:
                result = extractor(row["url"])
            except requests.RequestException as error:
                result = {
                    "body_status": "request_error",
                    "body_raw": "",
                    "body_clean": "",
                    "body_checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "error": str(error),
                }

            article = {
                "source_name": row["source_name"],
                "title": row["title"],
                "url": row["url"],
                "published": row["published"],
                "rss_summary": row["rss_summary"],
                "rss_content": json.loads(
                    row["rss_content"] or "[]"
                ),
                # 본문 처리로 RSS 확인 시각을 바꾸지 않습니다.
                "checked_at_utc": row["last_seen_utc"],
            }

            record = {**article, **result}

            # 기사별로 저장하여 완료된 결과를 남깁니다.
            save_collection([article], [record])

            log_name = datetime.now(timezone.utc).strftime(
                "%Y%m%dT%H%M%S_%fZ"
            )
            (TEST_DIR / f"body_attempt_{log_name}.json").write_text(
                json.dumps(record, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            print("본문 상태:", result["body_status"])
            print("본문 글자 수:", len(result.get("body_clean", "")))

            time.sleep(2)

    print("\n이번 본문 처리 완료")


print("공통 처리 함수 갱신 완료 — 아직 수집하지 않았습니다.")

In [ ]:
# 연결된 3개 매체에서 각각 최대 3건 처리

process_connected_bodies(limit_per_source=3)

In [ ]:
# The Guardian
# 주요 분야: 국제·정치·경제·기술

def extract_guardian_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    body = None
    selected = ""

    for selector in (
        ".article-body-commercial-selector",
        ".article-body-viewer-selector",
        ".content__article-body",
        '[itemprop="articleBody"]',
    ):
        candidate = soup.select_one(selector)
        if candidate is not None and candidate.select("p"):
            body = candidate
            selected = selector
            break

    if body is None:
        result["body_status"] = "no_body_container"
        return result

    result["body_raw"] = body.get_text("\n", strip=True)
    cleaned = BeautifulSoup(str(body), "html.parser")

    # 本문이 아닌 구조가 명확한 영역만 제외합니다.
    for tag in cleaned.select(
        "script, style, nav, aside, footer, form, button, "
        "figure, figcaption"
    ):
        tag.decompose()

    # 화면에서 숨긴 안내 요소를 제외합니다.
    for tag in cleaned.select('[aria-hidden="true"]'):
        tag.decompose()

    parts = []

    for tag in cleaned.select("p, h2, h3, ul, ol, blockquote"):
        if tag.find_parent(["p", "ul", "ol", "blockquote"]):
            continue

        text = tag.get_text(" ", strip=True)
        if text:
            parts.append(text)

    result["body_clean"] = "\n\n".join(parts)
    result["extraction_method"] = f"Guardian: {selected} v1"
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


guardian_candidates = [
    article
    for article in multi_articles
    if any(
        ref["source_name"] == "The Guardian"
        for ref in article["feed_refs"]
    )
    and not any(
        part in urlsplit(article["url"]).path.lower()
        for part in (
            "/video/", "/live/", "/audio/",
            "/gallery/", "/ng-interactive/",
        )
    )
]

guardian_results = []

for number, article in enumerate(guardian_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_guardian_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "body_checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "error": str(error),
        }

    guardian_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("추출 방식:", result.get("extraction_method", ""))
    print("본문 글자 수:", len(result["body_clean"]))

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

guardian_run = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)
guardian_file = TEST_DIR / f"guardian_body_check_{guardian_run}.json"

guardian_file.write_text(
    json.dumps(guardian_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n시험 기사:", len(guardian_results), "건")
print("결과 저장:", guardian_file)

In [ ]:
# Guardian 본문 함수 연결·표본 저장

BODY_EXTRACTORS["The Guardian"] = extract_guardian_body

save_collection(guardian_results, guardian_results)


# 공통 처리 함수가 사용하는 기사 경로 분류 규칙 갱신
def classify_body_request(source, url):
    path = urlsplit(url).path.lower()

    if source == "CNBC":
        if "/video/" in path:
            return "video_review"
        if "live-updates" in path:
            return "live_review"
        if any(
            part in path
            for part in ("/investingclub/", "/pro/")
        ):
            return "subscription_review"

    elif source == "BBC News":
        if "/videos/" in path or "/av/" in path:
            return "video_review"
        if "/live/" in path:
            return "live_review"
        if "/news/articles/" not in path:
            return "type_review"

    elif source == "The Guardian":
        if "/video/" in path:
            return "video_review"
        if "/live/" in path:
            return "live_review"
        if "/audio/" in path:
            return "audio_review"
        if any(
            part in path
            for part in ("/gallery/", "/ng-interactive/")
        ):
            return "type_review"

    return None


print("\n연결된 본문 추출 매체")
for source in BODY_EXTRACTORS:
    print("-", source)

print("\nGuardian 저장 상태")
with sqlite3.connect(TEST_DB) as conn:
    rows = conn.execute("""
        SELECT body_status, COUNT(*)
        FROM articles
        WHERE source_name = 'The Guardian'
        GROUP BY body_status
        ORDER BY body_status
    """).fetchall()

for status, count in rows:
    print(f"- {status}: {count}건")

In [ ]:
# TechCrunch
# 주요 분야: 기술·스타트업·벤처투자

def extract_techcrunch_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    body = None
    selected = ""

    # 페이지 전체가 아니라 첫 기사 본문 영역만 선택합니다.
    for selector in (
        ".entry-content",
        ".wp-block-post-content",
        '[itemprop="articleBody"]',
        ".article-content",
    ):
        candidate = soup.select_one(selector)

        if candidate is not None and candidate.select("p"):
            body = candidate
            selected = selector
            break

    if body is None:
        result["body_status"] = "no_body_container"
        return result

    result["body_raw"] = body.get_text("\n", strip=True)
    cleaned = BeautifulSoup(str(body), "html.parser")

    # 영상 자체·사진 설명·명확한 부가 영역을 제외합니다.
    for tag in cleaned.select(
        "script, style, nav, aside, footer, form, button, "
        "figure, figcaption, iframe, video"
    ):
        tag.decompose()

    parts = []

    for tag in cleaned.select("p, h2, h3, ul, ol, blockquote"):
        if tag.find_parent(["p", "ul", "ol", "blockquote"]):
            continue

        text = tag.get_text(" ", strip=True)
        if text:
            parts.append(text)

    result["body_clean"] = "\n\n".join(parts)
    result["extraction_method"] = f"TechCrunch: {selected} v1"
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


tc_candidates = [
    article
    for article in multi_articles
    if any(
        ref["source_name"] == "TechCrunch"
        for ref in article["feed_refs"]
    )
    and not any(
        part in urlsplit(article["url"]).path.lower()
        for part in ("/video/", "/podcast/", "/live/")
    )
]

tc_results = []

for number, article in enumerate(tc_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_techcrunch_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "body_checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "error": str(error),
        }

    tc_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("추출 방식:", result.get("extraction_method", ""))
    print("본문 글자 수:", len(result["body_clean"]))

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

tc_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
tc_file = TEST_DIR / f"techcrunch_body_check_{tc_run}.json"

tc_file.write_text(
    json.dumps(tc_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n시험 기사:", len(tc_results), "건")
print("결과 저장:", tc_file)

In [ ]:
# TechCrunch 본문 함수 연결·표본 저장

BODY_EXTRACTORS["TechCrunch"] = extract_techcrunch_body

save_collection(tc_results, tc_results)


# 기존 분류 규칙은 유지하고 TechCrunch 규칙을 추가합니다.
# 셀 재실행 시 함수가 중첩되지 않도록 한 번만 보관합니다.
if "_classify_before_techcrunch" not in globals():
    _classify_before_techcrunch = classify_body_request


def classify_body_request(source, url):
    if source == "TechCrunch":
        path = urlsplit(url).path.lower()

        if "/video/" in path:
            return "video_review"
        if "/podcast/" in path:
            return "audio_review"
        if "/live/" in path:
            return "live_review"

    return _classify_before_techcrunch(source, url)


print("\n연결된 본문 추출 매체")
for source in BODY_EXTRACTORS:
    print("-", source)

print("\nTechCrunch 저장 상태")

with sqlite3.connect(TEST_DB) as conn:
    rows = conn.execute("""
        SELECT body_status, COUNT(*)
        FROM articles
        WHERE source_name = 'TechCrunch'
        GROUP BY body_status
        ORDER BY body_status
    """).fetchall()

for status, count in rows:
    print(f"- {status}: {count}건")

In [ ]:
# MIT Technology Review
# 주요 분야: 신기술·AI·기술정책

def extract_mit_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")

    heading = soup.find("h1")
    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    # 이전 시험에서 본문 표시가 성공한 영역입니다.
    candidates = soup.select('[class*="contentBody__content--"]')

    # 중첩된 본문 영역은 중복 수집하지 않습니다.
    candidate_ids = {id(tag) for tag in candidates}
    blocks = [
        tag for tag in candidates
        if not any(
            id(parent) in candidate_ids
            for parent in tag.parents
        )
    ]

    if not blocks:
        result["body_status"] = "no_body_container"
        return result

    raw_parts = []
    clean_parts = []

    for block in blocks:
        raw_parts.append(block.get_text("\n", strip=True))
        cleaned = BeautifulSoup(str(block), "html.parser")

        # 명확한 부가 영역만 제거합니다.
        for tag in cleaned.select(
            "script, style, nav, aside, footer, form, button, "
            "figure, figcaption, iframe, video"
        ):
            tag.decompose()

        for tag in cleaned.select("p, h2, h3, ul, ol, blockquote"):
            if tag.find_parent(["p", "ul", "ol", "blockquote"]):
                continue

            text = tag.get_text(" ", strip=True)
            if text:
                clean_parts.append(text)

    result["body_raw"] = "\n\n".join(raw_parts)
    result["body_clean"] = "\n\n".join(clean_parts)
    result["extraction_method"] = "MIT: contentBody blocks v1"
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


mit_candidates = [
    article
    for article in multi_articles
    if any(
        ref["source_name"] == "MIT Technology Review"
        for ref in article["feed_refs"]
    )
]

mit_results = []

for number, article in enumerate(mit_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_mit_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "body_checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "error": str(error),
        }

    mit_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("본문 글자 수:", len(result["body_clean"]))

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

mit_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
mit_file = TEST_DIR / f"mit_body_check_{mit_run}.json"

mit_file.write_text(
    json.dumps(mit_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n시험 기사:", len(mit_results), "건")
print("결과 저장:", mit_file)

In [ ]:
# ABC News
# 주요 분야: 일반뉴스·경제·기술

def extract_abc_body(url):
    response = requests.get(
        url,
        headers={"User-Agent": "RSSNewsCollector/1.0"},
        timeout=(8, 25),
    )

    result = {
        "http_status": response.status_code,
        "final_url": response.url,
        "original_title": "",
        "body_raw": "",
        "body_clean": "",
        "body_status": "",
        "extraction_method": "",
        "removed_sections": [],
        "full_text_verified": False,
        "body_checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if response.status_code != 200:
        result["body_status"] = (
            "restricted"
            if response.status_code in (401, 402, 403, 451)
            else "http_error"
        )
        return result

    soup = BeautifulSoup(response.content, "html.parser")
    heading = soup.find("h1")

    if heading:
        result["original_title"] = heading.get_text(" ", strip=True)

    body = None
    selected = ""

    for selector in (
        '[data-testid="prism-article-body"]',
        ".Article__Content",
        ".article-copy",
        '[itemprop="articleBody"]',
    ):
        candidate = soup.select_one(selector)
        if candidate is not None and candidate.select("p"):
            body = candidate
            selected = selector
            break

    if body is None:
        result["body_status"] = "no_body_container"
        return result

    result["body_raw"] = body.get_text("\n", strip=True)
    cleaned = BeautifulSoup(str(body), "html.parser")

    # 추천 제목이 확인된 영역만 제거합니다.
    for section in list(
        cleaned.select('section[data-testid="prism-collection"]')
    ):
        section_heading = section.find(["h2", "h3"])

        if (
            section_heading
            and section_heading.get_text(
                " ", strip=True
            ).casefold() == "popular reads"
        ):
            result["removed_sections"].append(
                section.get_text(" ", strip=True)
            )
            section.decompose()

    # 추천 영역 뒤의 정상 본문은 그대로 유지합니다.
    for tag in cleaned.select(
        "script, style, nav, aside, footer, form, button, "
        "figure, figcaption, iframe, video"
    ):
        tag.decompose()

    parts = []

    for tag in cleaned.select("p, h2, h3, ul, ol, blockquote"):
        if tag.find_parent(["p", "ul", "ol", "blockquote"]):
            continue

        text = tag.get_text(" ", strip=True)
        if text:
            parts.append(text)

    result["body_clean"] = "\n\n".join(parts)
    result["extraction_method"] = f"ABC: {selected} v1"
    result["body_status"] = (
        "candidate_review" if result["body_clean"] else "no_body"
    )

    return result


abc_candidates = [
    article
    for article in multi_articles
    if any(
        ref["source_name"] == "ABC News"
        for ref in article["feed_refs"]
    )
    and not any(
        part in urlsplit(article["url"]).path.lower()
        for part in ("/video/", "/live/")
    )
]

abc_results = []

for number, article in enumerate(abc_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_abc_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "removed_sections": [],
            "body_checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "error": str(error),
        }

    abc_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("본문 글자 수:", len(result["body_clean"]))
    print("제거한 추천 영역:", len(result["removed_sections"]), "개")

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

abc_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
abc_file = TEST_DIR / f"abc_body_check_{abc_run}.json"

abc_file.write_text(
    json.dumps(abc_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n시험 기사:", len(abc_results), "건")
print("결과 저장:", abc_file)

In [ ]:
# ABC News 일반 기사만 선택해 다시 확인
# 잘못된 주소·선거 위젯·영상 등은 본문 요청에서 제외합니다.

from urllib.parse import urlsplit

def check_abc_article_url(url):
    try:
        parsed = urlsplit(url)
    except ValueError:
        return False, "주소 해석 실패"

    if parsed.scheme not in ("https", "http"):
        return False, "지원하지 않는 주소 형식"

    # 주소 중간에 다른 주소가 붙어 있는지 확인합니다.
    if "://" in url.split("://", 1)[-1]:
        return False, "주소가 중복 결합됨"

    if parsed.hostname not in {
        "abcnews.com",
        "www.abcnews.com",
        "abcnews.go.com",
        "www.abcnews.go.com",
    }:
        return False, "ABC 기사 도메인 아님"

    path = parsed.path.lower()

    if "/widgets/" in path:
        return False, "기사 아닌 위젯"

    if any(
        part in path
        for part in ("/video/", "/live/", "/photos/")
    ):
        return False, "별도 유형 검토 필요"

    # 현재 확인할 기사 주소 형식을 제한합니다.
    if "/story" not in path and "/wirestory/" not in path:
        return False, "일반 기사 경로 확인 필요"

    return True, ""


abc_candidates = []
abc_excluded = []

for article in multi_articles:
    if not any(
        ref["source_name"] == "ABC News"
        for ref in article["feed_refs"]
    ):
        continue

    valid, reason = check_abc_article_url(article["url"])

    if valid:
        abc_candidates.append(article)
    else:
        abc_excluded.append({
            "title": article["title"],
            "url": article["url"],
            "reason": reason,
        })

print("일반 기사 후보:", len(abc_candidates), "건")
print("요청 제외·검토 항목:", len(abc_excluded), "건")

for item in abc_excluded:
    print(f"- {item['reason']}: {item['title']}")

abc_results = []

for number, article in enumerate(abc_candidates[:3], 1):
    if number > 1:
        time.sleep(2)

    print("\n" + "=" * 60)
    print(f"[{number}] 제목:", article["title"])
    print("URL:", article["url"])

    try:
        result = extract_abc_body(article["url"])
    except requests.RequestException as error:
        result = {
            "body_status": "request_error",
            "body_raw": "",
            "body_clean": "",
            "removed_sections": [],
            "body_checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "error": str(error),
        }

    abc_results.append({**article, **result})

    print("HTTP:", result.get("http_status", "응답 없음"))
    print("상태:", result["body_status"])
    print("본문 글자 수:", len(result["body_clean"]))
    print(
        "제거한 추천 영역:",
        len(result.get("removed_sections", [])),
        "개",
    )

    if result.get("error"):
        print("오류:", result["error"])

    display(HTML(
        '<details style="margin:10px 0; border:1px solid #888; '
        'padding:12px;">'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        'white-space:pre-wrap; padding-top:12px;">'
        + html.escape(result["body_clean"] or "추출된 본문이 없습니다.")
        + "</div></details>"
    ))

abc_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
abc_file = TEST_DIR / f"abc_filtered_check_{abc_run}.json"

abc_file.write_text(
    json.dumps(
        {
            "results": abc_results,
            "excluded": abc_excluded,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n결과 저장:", abc_file)

In [ ]:
# ABC 첫 번째 기사 제목·URL·본문 다시 표시

if not abc_results:
    print("ABC 시험 결과가 없습니다.")
else:
    article = abc_results[0]
    title = html.escape(article.get("title") or "제목 없음")
    url = html.escape(article.get("url", ""), quote=True)
    body = html.escape(
        article.get("body_clean") or "추출된 본문이 없습니다."
    )

    display(HTML(
        '<div style="border:1px solid #888; padding:14px;">'
        f'<h3>ABC 1번 기사: {title}</h3>'
        f'<p><a href="{url}" target="_blank" '
        f'rel="noopener noreferrer">{url}</a></p>'
        '<details>'
        '<summary style="cursor:pointer;">본문 펼치기</summary>'
        '<div style="max-height:500px; overflow:auto; '
        f'white-space:pre-wrap; padding-top:12px;">{body}</div>'
        '</details></div>'
    ))

In [ ]:
# ABC 본문 함수 연결·검토 표본 저장

BODY_EXTRACTORS["ABC News"] = extract_abc_body
save_collection(abc_results, abc_results)


# 기존 매체 분류 규칙에 ABC 주소 검사를 추가합니다.
if "_classify_before_abc" not in globals():
    _classify_before_abc = classify_body_request


def classify_body_request(source, url):
    if source == "ABC News":
        valid, reason = check_abc_article_url(url)

        if not valid:
            if reason in {
                "주소 해석 실패",
                "지원하지 않는 주소 형식",
                "주소가 중복 결합됨",
                "ABC 기사 도메인 아님",
            }:
                return "invalid_url_review"

            path = urlsplit(url).path.lower()

            if "/widgets/" in path:
                return "non_article_review"
            if "/video/" in path:
                return "video_review"
            if "/live/" in path:
                return "live_review"

            return "type_review"

    return _classify_before_abc(source, url)


# 이미 저장된 ABC 대기 항목에도 같은 검사를 적용합니다.
with sqlite3.connect(TEST_DB) as conn:
    abc_pending = conn.execute("""
        SELECT url
        FROM articles
        WHERE source_name = 'ABC News'
          AND body_status = 'pending'
    """).fetchall()

    classified_counts = {}

    for (url,) in abc_pending:
        status = classify_body_request("ABC News", url)

        if status:
            conn.execute("""
                UPDATE articles
                SET body_status = ?
                WHERE url = ?
                  AND body_status = 'pending'
            """, (status, url))

            classified_counts[status] = (
                classified_counts.get(status, 0) + 1
            )

print("\n별도 검토로 분리한 항목")
for status, count in classified_counts.items():
    print(f"- {status}: {count}건")

print("\nABC 저장 상태")
with sqlite3.connect(TEST_DB) as conn:
    rows = conn.execute("""
        SELECT body_status, COUNT(*)
        FROM articles
        WHERE source_name = 'ABC News'
        GROUP BY body_status
        ORDER BY body_status
    """).fetchall()

for status, count in rows:
    print(f"- {status}: {count}건")

print("\n연결된 본문 추출 매체:", len(BODY_EXTRACTORS), "개")
for source in BODY_EXTRACTORS:
    print("-", source)

In [ ]:
# 통합 본문 수집 — 매체별 대기 기사 최대 1건 처리

import sys
import importlib
from pathlib import Path

src_path = Path(r"C:\ai_campus_main_proj\rss_news\src")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import integrated_notebook_bodies as integrated
importlib.reload(integrated)

# 매체별 규칙 적용 상태
integrated.show_report(integrated.source_rules())

# 저장된 대기 기사의 본문 처리
batch_summary, batch_folder = integrated.run_batch(
    limit_per_source=1,
    delay=2,
)

# 전체 결과표
integrated.show_report(batch_summary)

print("상세 결과 저장:", batch_folder)